In [1]:
%pip install -q langchain-google-genai pypdf langchain_community tiktoken langchain-ollama langchainhub chromadb langchain langchain-classic langchain-chroma

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")
if not os.environ('LANGCHAIN_API_KEY'):
    os.environ['LANGCHAIN_API_KEY'] = getpass.getpass("Enter LangSmith API Key: ")
    
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = 'rag-lab-foundation'

In [3]:
import os
print("Current working directory:", os.getcwd())
print("Contents of data/papers (if it exists):")
print(os.listdir("../data/papers") if os.path.exists("../data/papers") else "PATH DOES NOT EXIST")

Current working directory: /home/mohnish/Codes/rag-lab/notebooks
Contents of data/papers (if it exists):
['.ipynb_checkpoints', 'Active Learning with Physics-Informed Graph Neural\nNetworks on Unstructured Meshes.pdf', 'COMBINING PHYSICS-INFORMED GRAPH NEURAL\nNETWORK AND FINITE DIFFERENCE FOR SOLVING\nFORWARD AND INVERSE SPATIOTEMPORAL PDES.pdf', 'Distributed physics-informed neural networks via domain decomposition for fast flow reconstruction.pdf', 'LEARNING MESH-BASED SIMULATION WITH GRAPH NETWORKS.pdf', 'Learning to Simulate Complex Physics with Graph Networks.pdf', 'PINN for solving forward and inverse problems involving\nnonlinear partial differential equations.pdf', 'PINN-GNN Hybrid Neural Networks for Precise PUE Prediction in Data\nCenters.pdf', 'SK-PINN_ Accelerated physics-informed deep learning by smoothing kernel gradients.pdf', 'Towards Accelerating Physics Informed Graph Neural Network\nfor Fluid Simulation.pdf', 'When and Why PINNs fail to train: A Neural Network Persp

In [4]:
from pathlib import Path

# Resolves relative to this notebook's actual location, not the kernel's cwd
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PAPERS_DIR = PROJECT_ROOT / "data" / "papers"

print("Papers directory:", PAPERS_DIR)
print("Exists:", PAPERS_DIR.exists())
print("PDF count:", len(list(PAPERS_DIR.glob("*.pdf"))) if PAPERS_DIR.exists() else 0)

Papers directory: /home/mohnish/Codes/rag-lab/data/papers
Exists: True
PDF count: 11


In [5]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("/home/mohnish/Codes/rag-lab/data/papers")
raw_docs = loader.load()
print(f"{len(raw_docs)} pages loaded from corpus")

/tmp/ipykernel_110457/4107541281.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


258 pages loaded from corpus


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500, chunk_overlap=75
)
splits = splitter.split_documents(raw_docs)
print(f"{len(splits)} chunks")

576 chunks


In [7]:
import time
import re
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

def clean_surrogates(text: str) -> str:
    """Removes invalid UTF-16 surrogate code points from text."""
    if not isinstance(text, str):
        return text
    # Encode ignoring errors, then decode back to valid UTF-8
    return text.encode('utf-8', 'ignore').decode('utf-8')

# 1. Sanitize page content and metadata in your document splits
for doc in splits:
    doc.page_content = clean_surrogates(doc.page_content)
    
    # Sanitize metadata values like PDF titles or headers containing bad chars
    for key, value in doc.metadata.items():
        if isinstance(value, str):
            doc.metadata[key] = clean_surrogates(value)

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

BATCH_SIZE = 10
DELAY_SECONDS = 6   # keeps limit safely under the RPM ceiling

vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory="/home/mohnish/Codes/rag-lab/data/chroma_naive_gemini"
)

print("Embedding...")
for i in range(0, len(splits), BATCH_SIZE):
    batch = splits[i:i + BATCH_SIZE]
    vectorstore.add_documents(batch)
    time.sleep(DELAY_SECONDS)

print("Done.")
print(f"Embedded {len(splits)}.")

Embedding...
Done.
Embedded 576.


In [8]:
# Retrieval
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
test_query = "What is a Physics Informed Neural Network and how is it different from a Graphical Neural Network?"
docs = retriever.invoke(test_query)
for d in docs:
    print(d.metadata.get("source"), "-", d.page_content[:150], "\n")

/home/mohnish/Codes/rag-lab/data/papers/Active Learning with Physics-Informed Graph Neural.pdf - partial differential equations, Science Advances 3 (2017) e1602614. URL: https:
//www.science.org/doi/abs/10.1126/sciadv.1602614. doi: 10.1126/sciadv. 

/home/mohnish/Codes/rag-lab/data/papers/Active Learning with Physics-Informed Graph Neural
Networks on Unstructured Meshes.pdf - partial differential equations, Science Advances 3 (2017) e1602614. URL: https:
//www.science.org/doi/abs/10.1126/sciadv.1602614. doi: 10.1126/sciadv. 

/home/mohnish/Codes/rag-lab/data/papers/COMBINING PHYSICS-INFORMED GRAPH NEURAL.pdf - PIGNN 3
In addition to physics-based models, data-driven methods [22, 23, 24] also seek
nonlinear mappings from parametric DNNs to numerical solutions 

/home/mohnish/Codes/rag-lab/data/papers/COMBINING PHYSICS-INFORMED GRAPH NEURAL.pdf - PIGNN 3
In addition to physics-based models, data-driven methods [22, 23, 24] also seek
nonlinear mappings from parametric DNNs to numerical so

In [9]:
# Generation

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatGoogleGenerativeAI(model="models/gemini-3.6-flash", temperature=0)

template = """Answer the question based only on the following context:
{context}

Question: {question}"""
prompt = ChatPromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print(rag_chain.invoke(test_query))

/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Based on the provided documents:

* **Physics-Informed Neural Network (PINN):** It is a deep learning framework used for solving forward and inverse problems involving (nonlinear) partial differential equations (PDEs) by integrating physical information into the neural network's learning process.

* **Graph Neural Network (GNN) / Key Differences:** Rather than focusing strictly on physics-based constraints, a GNN models relationships and structures between nodes. It uses a **message passing mechanism** to perform local information mining and relational reasoning. GNNs are specifically suited for describing unstructured mesh data by establishing a one-to-one mapping between mesh points and graph nodes.


In [10]:
from rag_lab.strategies.naive import NaiveStrategy
strategy = NaiveStrategy(vectorstore, llm=llm) 
print(strategy.run(test_query))

/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided context, the two concepts are described as follows:

* **Physics-Informed Neural Network (PINN):** 
  According to the context, a Physics-Informed Neural Network (PINN) is a deep learning framework used for solving forward and inverse problems involving (nonlinear or spatiotemporal) partial differential equations (PDEs). It works by integrating physical information into the model, allowing it to learn mesh-based or numerical solutions to PDEs with few or no pre-computed labeled data.

* **Graph Neural Network (GNN):** 
  A Graph Neural Network (GNN) is a network designed to model relational structure between nodes using a **message passing mechanism**. It performs local information mining and relational reasoning. GNNs are used to describe unstructured mesh data or particle dynamics by establishing a one-to-one mapping between mesh points and graph nodes.

### Difference based on the text:
* **Function and Structure:** GNNs specifically rely on graph structures an